# Simulasi Pelacakan Sinar (Ray-Tracing) pada Ruang-Waktu Kerr-de Sitter

Notebook ini berisi pemodelan lintasan foton di sekitar lubang hitam bermassa $M$, spin $a$, dan konstanta kosmologis $\Lambda$. Kode telah direfaktor menggunakan praktik terbaik *(best practices)* Python seperti _type-hinting_, _dataclass_ untuk konfigurasi, dan modularitas fungsi, sehingga mempermudah pembacaan dan pemeliharaan saat dipublikasikan (contohnya di GitHub).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from abc import ABC, abstractmethod
from typing import Tuple, List, Dict
import logging
from dataclasses import dataclass

# Konfigurasi logging untuk memudahkan proses debugging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')


## 1. Abstraksi Kelas (Base Classes)

Menggunakan kelas abstrak menjamin bahwa setiap metrik ruang-waktu maupun integrator numerik mematuhi _interface_ (struktur) yang baku.


In [ ]:
class Spacetime(ABC):
    """Abstract Base Class untuk mendefinisikan sifat geometri ruang-waktu (Hamiltonian)."""
    
    @abstractmethod
    def H(self, x: np.ndarray, p: np.ndarray) -> float:
        """Menghitung nilai Hamiltonian pada titik fasa (x, p)."""
        pass

    @abstractmethod
    def dHdx(self, x: np.ndarray, p: np.ndarray) -> np.ndarray:
        """Menghitung turunan parsial Hamiltonian terhadap koordinat spasial (x)."""
        pass

    @abstractmethod
    def dHdp(self, x: np.ndarray, p: np.ndarray) -> np.ndarray:
        """Menghitung turunan parsial Hamiltonian terhadap momentum (p)."""
        pass

class Integrator(ABC):
    """Abstract Base Class untuk metode penyelesaian persamaan diferensial (integrator)."""
    
    @abstractmethod
    def step(self, x: np.ndarray, p: np.ndarray, dt: float) -> Tuple[np.ndarray, np.ndarray]:
        """Melakukan iterasi waktu (dt) satu langkah ke depan."""
        pass


## 2. Integrator Numerik (Runge-Kutta Orde 4)

Metode klasik untuk mengintegrasikan sistem persamaan Hamiltonian.


In [ ]:
class RungeKutta4(Integrator):
    """
    Penyelesaian Sistem Hamiltonian menggunakan algoritma Runge-Kutta Orde 4 (RK4).
    """
    def __init__(self, spacetime: Spacetime):
        self.spacetime = spacetime

    def step(self, x: np.ndarray, p: np.ndarray, dt: float) -> Tuple[np.ndarray, np.ndarray]:
        """
        Menghitung iterasi RK4 untuk memajukan posisi dan momentum.
        
        Args:
            x (np.ndarray): Array posisi [t, r, theta, phi]
            p (np.ndarray): Array momentum [p_t, p_r, p_theta, p_phi]
            dt (float): Ukuran langkah waktu integrasi (dtau)
            
        Returns:
            Tuple[np.ndarray, np.ndarray]: Posisi dan momentum di langkah berikutnya.
        """
        # Step 1
        dHp1 = self.spacetime.dHdp(x, p)
        dHx1 = self.spacetime.dHdx(x, p)
        k1 = -dHx1 * dt
        l1 = dHp1 * dt

        # Step 2
        dHp2 = self.spacetime.dHdp(x + 0.5*l1, p + 0.5*k1)
        dHx2 = self.spacetime.dHdx(x + 0.5*l1, p + 0.5*k1)
        k2 = -dHx2 * dt
        l2 = dHp2 * dt

        # Step 3
        dHp3 = self.spacetime.dHdp(x + 0.5*l2, p + 0.5*k2)
        dHx3 = self.spacetime.dHdx(x + 0.5*l2, p + 0.5*k2)
        k3 = -dHx3 * dt
        l3 = dHp3 * dt

        # Step 4
        dHp4 = self.spacetime.dHdp(x + l3, p + k3)
        dHx4 = self.spacetime.dHdx(x + l3, p + k3)
        k4 = -dHx4 * dt
        l4 = dHp4 * dt

        p_new = p + (k1 + 2*k2 + 2*k3 + k4) / 6.0
        x_new = x + (l1 + 2*l2 + 2*l3 + l4) / 6.0

        return x_new, p_new


## 3. Fisika Metrik (Kerr-De Sitter Spacetime)

Metrik diformulasikan menggunakan _SymPy_ untuk mendapatkan penurunan secara otomatis sebelum dieksekusi secara numerik melalui `lambdify`. Pendekatan ini mengurangi _human error_ saat menurunkan komponen Hamiltonian.


In [ ]:
class KerrDeSitter(Spacetime):
    """
    Kelas fisika yang merepresentasikan metrik Kerr-de Sitter di koordinat Boyer-Lindquist.
    """
    def __init__(self, M: float, a: float, lambda_cosmo: float):
        self.M_val = M
        self.a_val = a
        self.lambda_val = lambda_cosmo
        
        logging.info(f"Menginisialisasi geometri: M={M}, a={a}, Lambda={lambda_cosmo}")
        self._derive_symbolic_equations()

    def _derive_symbolic_equations(self):
        """Menurunkan persamaan diferensial secara analitik (Symbolic Math)."""
        # 1. Definisi Simbol
        t, r, th, ph = sp.symbols('t r theta phi')
        pt, pr, pth, pph = sp.symbols('pt pr ptheta pphi')
        M, a, L = sp.symbols('M a Lambda')

        # 2. Fungsi Metrik (Boyer-Lindquist) Kerr-de Sitter
        Sigma = r**2 + (a * sp.cos(th))**2
        Delta_r = (r**2 + a**2) * (1 - (L * r**2)/3) - 2*M*r
        Delta_th = 1 + (L * a**2 * sp.cos(th)**2)/3
        Xi = 1 + (L * a**2)/3

        # Ekstrak komponen metrik kovarian g_uv
        g_tt = - (Delta_r / Sigma) + (Delta_th * sp.sin(th)**2 / Sigma) * a**2
        g_rr = Sigma / Delta_r
        g_thth = Sigma / Delta_th
        g_pp = -(Delta_r/Sigma) * (a * sp.sin(th)**2 / Xi)**2 + \
               (Delta_th * sp.sin(th)**2 / Sigma) * ((r**2 + a**2)/Xi)**2
        g_tp = -(Delta_r/Sigma) * (-a * sp.sin(th)**2 / Xi) + \
               (Delta_th * sp.sin(th)**2 / Sigma) * (-a * (r**2 + a**2)/Xi)

        # Matriks & Invers
        g_mat = sp.Matrix([
            [g_tt, 0, 0, g_tp],
            [0, g_rr, 0, 0],
            [0, 0, g_thth, 0],
            [g_tp, 0, 0, g_pp]
        ])
        g_inv = g_mat.inv()

        # 3. Hamiltonian H = 1/2 g^uv p_u p_v
        p_vec = sp.Matrix([pt, pr, pth, pph])
        self.Ham_expr = sp.simplify(0.5 * (p_vec.T * g_inv * p_vec)[0])

        # 4. Turunan Hamiltonian
        vars_x = [t, r, th, ph]
        vars_p = [pt, pr, pth, pph]
        dHdx_expr = [sp.diff(self.Ham_expr, v) for v in vars_x]
        dHdp_expr = [sp.diff(self.Ham_expr, v) for v in vars_p]

        # 5. Konversi ke bentuk fungsi NumPy (Compiled Level Speed)
        params = (t, r, th, ph, pt, pr, pth, pph, M, a, L)
        self.f_H = sp.lambdify(params, self.Ham_expr, 'numpy')
        self.f_dHdx = sp.lambdify(params, dHdx_expr, 'numpy')
        self.f_dHdp = sp.lambdify(params, dHdp_expr, 'numpy')
        self.f_g_inv = sp.lambdify(params[:4] + (M, a, L), g_inv, 'numpy')

    def _unpack_state(self, x: np.ndarray, p: np.ndarray) -> Tuple:
        return x[0], x[1], x[2], x[3], p[0], p[1], p[2], p[3]

    def H(self, x: np.ndarray, p: np.ndarray) -> float:
        args = (*self._unpack_state(x, p), self.M_val, self.a_val, self.lambda_val)
        return self.f_H(*args)

    def dHdx(self, x: np.ndarray, p: np.ndarray) -> np.ndarray:
        args = (*self._unpack_state(x, p), self.M_val, self.a_val, self.lambda_val)
        return np.array(self.f_dHdx(*args))

    def dHdp(self, x: np.ndarray, p: np.ndarray) -> np.ndarray:
        args = (*self._unpack_state(x, p), self.M_val, self.a_val, self.lambda_val)
        return np.array(self.f_dHdp(*args))

    def get_initial_pr(self, x: np.ndarray, E: float, L_ang: float) -> float:
        """Mencari nilai momentum radial awal (pr) untuk null geodesics (H=0)."""
        t, r, th, ph = x
        args = (t, r, th, ph, self.M_val, self.a_val, self.lambda_val)
        g_inv = self.f_g_inv(*args)

        gUU_tt, gUU_rr = g_inv[0,0], g_inv[1,1]
        gUU_tp, gUU_pp = g_inv[0,3], g_inv[3,3]

        if abs(gUU_rr) < 1e-10:
            raise ValueError(f"Singularitas koordinat terdeteksi (g^rr sangat kecil): {gUU_rr} di r={r}")

        residue = gUU_tt * E**2 + gUU_pp * L_ang**2 + 2 * gUU_tp * (-E) * L_ang
        pr_sq = -residue / gUU_rr

        if pr_sq < 0:
            raise ValueError(f"Area terlarang foton: pr^2 = {pr_sq} < 0 (E={E}, L={L_ang})")

        return -np.sqrt(pr_sq)


## 4. Konfigurasi Eksperimen

Mengumpulkan seluruh konstanta dan parameter setup di dalam satu struktur dataclass agar memudahkan _hyperparameter tuning_ atau pengubahan parameter eksperimen.


In [ ]:
@dataclass
class SimulationConfig:
    """
    Struktur data untuk parameter awal eksperimen ray-tracing.
    """
    M: float = 1.0                     # Massa Lubang Hitam
    a: float = -0.99                   # Parameter Spin
    lambdas: List[float] = None        # Array Konstanta Kosmologis (Lambda)
    r0: float = 500.0                  # Jarak awal radiasi (Source)
    theta0: float = np.pi / 2          # Sudut polar awal (di bidang ekuator)
    phi0: float = 0.0                  # Sudut azimuth awal
    E: float = 1.0                     # Energi Foton
    b: float = 7.0                     # Impact Parameter
    dtau: float = 0.05                 # Step size integrasi afin
    max_steps: int = 30000             # Maksimum perulangan integrasi

    def __post_init__(self):
        # Inisialisasi default jika lambdas tidak disertakan
        if self.lambdas is None:
            self.lambdas = [0.0, -1e-8, -5e-8, -1e-7, -5e-7, -1e-6, -5e-6]
            
        self.L_ang = self.b * self.E                  # Momentum sudut L
        self.rh = self.M + np.sqrt(self.M**2 - self.a**2) # Horizon radius (Kerr approx)

# Instantiate Konfigurasi
config = SimulationConfig()


## 5. Mesin Simulasi Utama

Menjalankan perulangan komputasi sebagai sebuah fungsi independen (`run_simulation`) alih-alih meletakkannya di *global scope*. Ini sangat penting untuk manajemen memori dan mempermudah Debugging.


In [ ]:
def run_simulation(cfg: SimulationConfig) -> List[Dict]:
    """
    Fungsi utama untuk menjalankan integrasi lintasan (ray-tracing) 
    untuk seluruh variasi nilai Konstanta Kosmologis.
    """
    results = []
    
    print(f"{'a':<10} | {'Lambda':<10} | {'Defleksi (deg)':<20} | {'Status'}")
    print("-" * 60)

    for lmd in cfg.lambdas:
        spacetime = KerrDeSitter(cfg.M, cfg.a, lmd)
        integrator = RungeKutta4(spacetime)
        
        # Kondisi awal posisi dan momentum
        x = np.array([0.0, cfg.r0, cfg.theta0, cfg.phi0])
        try:
            pr_init = spacetime.get_initial_pr(x, cfg.E, cfg.L_ang)
        except ValueError as err:
            logging.warning(f"Melewati L={lmd}: {err}")
            continue

        p = np.array([-cfg.E, pr_init, 0.0, cfg.L_ang])
        
        steps = 0
        X_vals, Y_vals, Phi_vals, H_vals = [], [], [], []

        # Loop Integrasi
        while steps < cfg.max_steps:
            x, p = integrator.step(x, p, cfg.dtau)
            r_curr, phi_curr = x[1], x[3]

            # Mencatat state
            H_vals.append(spacetime.H(x, p))
            X_vals.append(r_curr * np.cos(phi_curr))
            Y_vals.append(r_curr * np.sin(phi_curr))
            Phi_vals.append(phi_curr)

            # --- Stop Conditions (Kriteria Penghentian) ---
            if r_curr < cfg.rh * 1.1:
                logging.debug(f"Sinar L={lmd} ditelan Black Hole pada step {steps}.")
                break
            if r_curr > cfg.r0 + 5 and steps > 10:
                logging.debug(f"Sinar L={lmd} keluar asimtot pada step {steps}.")
                break

            steps += 1

        # Analisis Defleksi & Rekapitulasi Data
        if len(Phi_vals) > 0:
            deflection = abs(abs(Phi_vals[-1] - cfg.phi0) - np.pi)
            def_deg = np.degrees(deflection)
            print(f"{cfg.a:<10} | {lmd:<10} | {def_deg:<20.4f} | Selesai (Steps: {steps})")
            
            results.append({
                'lambda': lmd,
                'deflection': def_deg,
                'X': X_vals,
                'Y': Y_vals,
                'H': H_vals
            })
            
    return results

# Eksekusi Simulasi
sim_results = run_simulation(config)


## 6. Modul Plot & Visualisasi Data

Fungsi-fungsi terpisah untuk visualisasi lintasan dan analitik numerik.


In [ ]:
def plot_trajectories(results: List[Dict], cfg: SimulationConfig):
    """Visualisasi lintasan sinar secara spasial."""
    fig, ax = plt.subplots(figsize=(8,8))
    
    # Colormap yang aman bagi penderita buta warna (color-blind friendly)
    colors = plt.cm.viridis(np.linspace(0, 1, len(results)))

    # Horizon Estimasi (Abu-abu transparan)
    horizon = plt.Circle((0,0), cfg.rh, color='gray', alpha=0.3, label="Horizon (Kerr)")
    ax.add_artist(horizon)
    ax.scatter(0, 0, s=40, c='black', marker='x', label='Singularitas')

    for i, res in enumerate(results):
        label_txt = rf"$\Lambda = {res['lambda']}$"
        ax.plot(res['X'], res['Y'], color=colors[i], label=label_txt, linewidth=1.5)

    ax.set_title(f"Lintasan Sinar Kerr-De Sitter (M={cfg.M}, a={cfg.a}, b={cfg.b})", fontsize=14)
    ax.set_xlabel(r'$X = r \cos(\phi)$', fontsize=12)
    ax.set_ylabel(r'$Y = r \sin(\phi)$', fontsize=12)
    ax.set_aspect('equal')
    
    limit = 10
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    
    # Legend diletakkan di luar plot agar tidak menutupi lintasan
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# Panggil fungsi plot
if sim_results:
    plot_trajectories(sim_results, config)


In [ ]:
def plot_hamiltonian_conservation(result: Dict):
    """Validasi konservasi Hamiltonian dari sebuah lintasan tunggal untuk memonitor error numerik."""
    H_arr = np.array(result['H'])
    r_arr = np.sqrt(np.array(result['X'])**2 + np.array(result['Y'])**2)
    steps_arr = np.arange(len(H_arr))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- Plot 1: Fluktuasi H vs Langkah (Waktu Integrasi) ---
    ax1.plot(steps_arr, H_arr, color='teal', label=r'Hamiltonian $\mathcal{H}$')
    ax1.set_yscale('symlog', linthresh=1e-12)
    ax1.set_title("Evolusi Hamiltonian terhadap Waktu Integrasi")
    ax1.set_xlabel("Langkah (Steps)")
    ax1.set_ylabel(r"Nilai $\mathcal{H}$")
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()
    
    # --- Plot 2: H vs Jarak Radial (r) ---
    ax2.plot(r_arr, H_arr, color='coral', label=r'Hamiltonian $\mathcal{H}$')
    ax2.axhline(0, color='red', linestyle=':', linewidth=1.5, label='Ideal $\mathcal{H}=0$')
    ax2.set_yscale('symlog', linthresh=1e-12)
    
    # Anotasi Titik Periapsis (Titik terdalam dari black hole)
    min_r_idx = np.argmin(r_arr)
    min_r = r_arr[min_r_idx]
    ax2.annotate(
        f'Periapsis\n$r \approx {min_r:.2f}$', 
        xy=(min_r, H_arr[min_r_idx]),
        xytext=(min_r + 2, H_arr[min_r_idx] + 1e-11),
        arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
    )
                 
    ax2.set_title("Konservasi Hamiltonian vs Jarak Radial ($r$)")
    ax2.set_xlabel("Koordinat Radial ($r$)")
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend()
    
    plt.suptitle(rf"Validasi Integrator Numerik RK4 ($\Lambda = {result['lambda']}$)", fontsize=15, y=1.05)
    plt.tight_layout()
    plt.show()

# Panggil validasi untuk data terakhir (Lambda paling besar dampaknya)
if sim_results:
    plot_hamiltonian_conservation(sim_results[-1])
